In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Example input
label = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0])
win_size = 4

def transform(label, win_size):

    # End result of this function:
    # - All positions where the corresponding context window contains at least one anomaly (indicated by 1) will be set to 2 (except for the positions that are originally 1, which will remain 1).
    # - The first and last few positions for which the window cannot be fully applied (i.e., the anomaly score is padded) will be set to -1.

    # Get indices of ones
    ones_indices = np.where(label == 1)[0]

    # Initialize a boolean mask for positions to be set to 2
    mask = np.zeros_like(label, dtype=bool)

    # Mark all indices within threshold distance of a 1
    for idx in ones_indices:
        start = max(0, idx + win_size // 2 + 1 - win_size)
        end = min(len(label), idx + win_size // 2 + 1)
        mask[start:end] = True

    # Change only the zeros within the mask to twos
    result = label.copy()
    result[(label == 0) & mask] = 2

    # Mask the first and last few positions for which the window cannot be fully applied (i.e., the anomaly score is padded)
    result[:win_size // 2] = -1
    if win_size > 2:
        result[-(win_size - win_size // 2) + 1:] = -1

    return result

# Example output
transform(label, win_size)

In [ ]:
model_size = 'large' # can be 'base' or 'large'
win_size = 1024 # can be 64, 256, 512, or 1024 (only for 'large')

mses_1 = np.array([])
mses_2 = np.array([]) # not used
mses_3 = np.array([])
mses_4 = np.array([]) # not used

# This is the epsilon value used in the implementation of MOMENT for z-normalization 
# (See `revin.py` in their repository: https://github.com/moment-timeseries-foundation-model/moment).
eps = 1e-5

data_direc = '../data/TSB-AD-U'
score_direc = f'../results/intermediate/moment_{model_size}_{win_size}/scores'

file_names = pd.read_csv('../data/File_List/TSB-AD-U-Eva.csv')['file_name'].to_numpy()
# If using the contamination-free subset (for results in Appendix G), use the following line instead to get the file names:
# file_names = pd.read_csv('moment_contamination_free.csv')['file_name'].to_numpy()

for file_name in file_names:

    # This time series is excluded for the 1024 window size because it is shorter than 1024.
    if win_size == 1024:
        if file_name == '010_NAB_id_10_WebService_tr_500_1st_271.csv':
            continue

    scores = np.load(f'{score_direc}/{os.path.splitext(file_name)[0]}.npy')
    df = pd.read_csv(f'{data_direc}/{file_name}').dropna()
    label = df['Label'].astype(int).to_numpy()
    label = transform(label, win_size)
    std = df['Data'].rolling(window = win_size, center = True).std().bfill().ffill().to_numpy()

    mask_1 = (label == 0) & (std != 0) # corresponds to context windows containing NO anomalies and having non-zero standard deviation.
    mask_2 = (label == 0) & (std == 0) # not used
    mask_3 = ((label == 1) | (label == 2)) & (std != 0) # corresponds to context windows containing anomalies and having non-zero standard deviation.
    mask_4 = ((label == 1) | (label == 2)) & (std == 0) # not used

    # Calculate MSEs for the normalized context windows, based on the anomaly scores,
    # which are just MSEs of the unnormalized context windows. 
    mses_1 = np.concatenate([mses_1, scores[mask_1] / ((std[mask_1] + eps) ** 2)])
    mses_2 = np.concatenate([mses_2, scores[mask_2] / ((std[mask_2] + eps) ** 2)]) # not used
    mses_3 = np.concatenate([mses_3, scores[mask_3] / ((std[mask_3] + eps) ** 2)])
    mses_4 = np.concatenate([mses_4, scores[mask_4] / ((std[mask_4] + eps) ** 2)]) # not used
    
# Visualization with histograms

weights_1 = np.ones_like(mses_1) * 100.0 / mses_1.size
weights_3 = np.ones_like(mses_3) * 100.0 / mses_3.size

plt.figure(figsize=(6, 6))

plt.hist([mses_1, mses_3],
         bins=10,
         range=(0, 2),
         weights=[weights_1, weights_3],
         label=['Normal', 'Anomalous'],
         edgecolor='black',
         alpha=0.7)

font_size = 14

plt.xlabel("MSE", fontsize=font_size)
plt.ylabel("Percent (%)", fontsize=font_size)
plt.legend(fontsize=font_size)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
#plt.ylim(0, 55)
plt.show()